# Run and analyse 48-beam optimisations

This notebook runs a selected pyDART CPU or GPU optimisation in an isolated subprocess, then reports convergence, timings, optimisation history, the overall-best simulation and parameters, and the best design from every completed restart.

Run it from any directory. GPU mode verifies that JAX can see a GPU before starting the expensive optimisation.

In [ ]:
%matplotlib inline

# Choose any combination of "cpu" and "gpu".
RUN_TARGETS = ("gpu",)
RUN_OPTIMISATIONS = True

# These may be replaced with paths to other optimisation TOML files.
CONFIGURATIONS = {
    "gpu": "configs/optimisations/generic_48_beam_design.toml",
    "cpu": "configs/optimisations/generic_48_beam_design_cpu.toml",
}

SHOW_RESTART_SIMULATIONS = True
SHOW_RESTART_PARAMETERS = True
FIGURE_DPI = 100

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not find the pyDART repository root.")


ROOT = find_repository_root(Path.cwd())
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from pydart.config import load_optimisation_config
from pydart.optimisation.persistence import load_optimisation_checkpoint
from pydart.plotting.optimisation import plot_optimisation_history
from plot_optimisation_animation import (
    discover_restart_snapshots,
    load_history,
    load_saved_snapshot,
    render_frame,
)
from plot_optimisation_parameters import (
    load_best_parameters,
    load_restart_parameters,
    plot_parameter_summary,
)

config_paths = {
    target: (ROOT / CONFIGURATIONS[target]).resolve()
    for target in RUN_TARGETS
}
for target, path in config_paths.items():
    if target not in {"cpu", "gpu"}:
        raise ValueError(f"Unknown target {target!r}; use 'cpu' or 'gpu'.")
    if not path.is_file():
        raise FileNotFoundError(path)

print(f"Repository: {ROOT}")
print("Selected configurations:")
for target, path in config_paths.items():
    print(f"  {target}: {path.relative_to(ROOT)}")

In [ ]:
def subprocess_environment(target: str) -> dict[str, str]:
    environment = os.environ.copy()
    if target == "cpu":
        environment["JAX_PLATFORMS"] = "cpu"
    else:
        # Let JAX select CUDA or ROCm according to the installed accelerator build.
        environment.pop("JAX_PLATFORMS", None)
    return environment


def available_devices(target: str) -> list[str]:
    probe = (
        "import json, jax; "
        "print(json.dumps([f'{d.platform}:{d.device_kind}' for d in jax.devices()]))"
    )
    completed = subprocess.run(
        [sys.executable, "-c", probe],
        cwd=ROOT,
        env=subprocess_environment(target),
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(completed.stdout.strip().splitlines()[-1])


for target in RUN_TARGETS:
    devices = available_devices(target)
    print(f"{target.upper()} devices: {', '.join(devices)}")
    if target == "gpu" and not any(
        device.startswith(("gpu:", "cuda:", "rocm:")) for device in devices
    ):
        raise RuntimeError(
            "GPU mode was selected, but the optimisation subprocess cannot see a "
            "GPU. Install the accelerator-enabled JAX build before continuing."
        )

In [ ]:
if RUN_OPTIMISATIONS:
    for target, config_path in config_paths.items():
        print(f"\n{'=' * 20} Starting {target.upper()} run {'=' * 20}", flush=True)
        completed = subprocess.run(
            [sys.executable, "-m", "pydart.cli.optimise", str(config_path)],
            cwd=ROOT,
            env=subprocess_environment(target),
            check=False,
        )
        if completed.returncode:
            print(
                f"WARNING: {target.upper()} optimisation exited with code "
                f"{completed.returncode}. Any checkpoint it produced will still "
                "be analysed."
            )

In [ ]:
def run_artifacts(config_path: Path) -> dict:
    config = load_optimisation_config(config_path)
    run_directory = (
        config.run.output_directory / f"optimisation_{config.run.index}"
    )
    checkpoint = run_directory / (
        f"optimisation_checkpoint_{config.run.index}.h5"
    )
    summary_path = run_directory / (
        f"optimisation_summary_{config.run.index}.json"
    )
    timing_path = run_directory / (
        f"optimisation_timing_{config.run.index}.json"
    )
    missing = [
        path for path in (checkpoint, summary_path, timing_path)
        if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing run artifacts: " + ", ".join(str(path) for path in missing)
        )
    return {
        "config": config,
        "directory": run_directory,
        "checkpoint": checkpoint,
        "summary": json.loads(summary_path.read_text(encoding="utf-8")),
        "timing": json.loads(timing_path.read_text(encoding="utf-8")),
    }


runs = {
    target: run_artifacts(config_path)
    for target, config_path in config_paths.items()
}
print("Loaded:", ", ".join(
    f"{target} -> {data['directory']}" for target, data in runs.items()
))

## Convergence, best result, and timing

In [ ]:
def print_run_summary(target: str, data: dict) -> None:
    summary = data["summary"]
    restart_results = summary["restart_results"]
    converged = sum(bool(result["success"]) for result in restart_results)
    objectives = np.asarray(
        [result["best_objective"] for result in restart_results], dtype=float
    )
    near_best = np.isclose(
        objectives,
        float(summary["best_objective"]),
        rtol=0.01,
        atol=1.0e-8,
    )
    print(f"\n{target.upper()} — {data['directory']}")
    print(f"  solver:                     {summary['solver']}")
    print(
        f"  converged restarts:         {converged}/"
        f"{len(restart_results)}"
    )
    print(f"  within 1% of global best:   {int(near_best.sum())}/{len(near_best)}")
    print(f"  elapsed optimisation time:  {summary['elapsed_seconds']:.3f} s")
    print(f"  overall success:            {summary['success']}")
    print(f"  best restart:               {summary['best_restart']}")
    print(f"  best objective:             {summary['best_objective']:.8e}")
    print(
        "  best RMS nonuniformity:     "
        f"{summary['best_rms_nonuniformity']:.8e}"
    )
    print(
        "  best deposited capacity:    "
        f"{summary['best_deposited_capacity_fraction']:.8e}"
    )
    print("\n  Restart  Conv.  Iterations  Evaluations  Best objective")
    for result in restart_results:
        print(
            f"  {result['restart_index']:>7}  "
            f"{str(bool(result['success'])):>5}  "
            f"{result['iterations']:>10}  "
            f"{result['function_evaluations']:>11}  "
            f"{result['best_objective']:.8e}"
        )


for target, data in runs.items():
    print_run_summary(target, data)

In [ ]:
for target, data in runs.items():
    sections = data["timing"]["sections"]
    names = list(sections)
    seconds = np.asarray([sections[name]["total_seconds"] for name in names])
    figure, axis = plt.subplots(figsize=(9, max(3, 0.55 * len(names))))
    axis.barh(
        [name.replace("_", " ") for name in names],
        seconds,
        color="steelblue",
    )
    axis.set(
        title=(
            f"{target.upper()} timing breakdown — "
            f"{data['timing']['total_seconds']:.2f} s total"
        ),
        xlabel="Wall time (s)",
    )
    axis.grid(axis="x", alpha=0.25)
    display(figure)
    plt.close(figure)

## Optimisation histories

In [ ]:
for target, data in runs.items():
    records, restart_results, best_record, elapsed = (
        load_optimisation_checkpoint(data["checkpoint"])
    )
    figure = plot_optimisation_history(records)
    figure.suptitle(
        f"{target.upper()} — optimisation history "
        f"({len(restart_results)} completed restarts)",
        fontsize=16,
    )
    display(figure)
    plt.close(figure)

## Overall-best simulation and parameters

In [ ]:
for target, data in runs.items():
    run_directory = data["directory"]
    history = load_history(run_directory)
    snapshot = load_saved_snapshot(run_directory / "best_simulation")
    output = run_directory / "notebook_plots" / "overall_best_simulation.png"
    render_frame(
        history,
        snapshot,
        int(history.history_index[-1]),
        output,
        dpi=FIGURE_DPI,
        design_label=(
            f"{target.upper()} overall best from restart "
            f"{snapshot.restart_index}"
        ),
    )
    print(f"{target.upper()} overall-best simulation")
    display(Image(filename=str(output)))

    parameters, objective = load_best_parameters(data["checkpoint"])
    figure = plot_parameter_summary(
        parameters,
        objective,
        design_label=f"{target.upper()} overall-best parameters",
    )
    display(figure)
    plt.close(figure)

## Best simulation and parameters from every restart

These cells use the stable restart snapshots written at the end of each completed restart. Disable either display in the settings cell if a compact report is preferable.

In [ ]:
for target, data in runs.items():
    run_directory = data["directory"]
    history = load_history(run_directory)
    snapshots = discover_restart_snapshots(run_directory)
    parameter_results = {
        restart: (parameters, objective)
        for restart, parameters, objective in load_restart_parameters(
            data["checkpoint"]
        )
    }

    print(f"\n{'=' * 18} {target.upper()} restart bests {'=' * 18}")
    for snapshot in snapshots:
        restart = snapshot.restart_index
        print(f"\n{target.upper()} restart {restart}")

        if SHOW_RESTART_SIMULATIONS:
            output = (
                run_directory
                / "notebook_plots"
                / f"restart_{restart}_best_simulation.png"
            )
            render_frame(
                history,
                snapshot,
                snapshot.history_index,
                output,
                dpi=FIGURE_DPI,
                design_label=f"{target.upper()} restart {restart} best",
            )
            display(Image(filename=str(output)))

        if SHOW_RESTART_PARAMETERS:
            parameters, objective = parameter_results[restart]
            figure = plot_parameter_summary(
                parameters,
                objective,
                design_label=f"{target.upper()} restart {restart} best parameters",
            )
            display(figure)
            plt.close(figure)

## CPU/GPU comparison

When both targets are selected, this final chart compares their per-restart best objectives and total recorded run time.

In [ ]:
if len(runs) > 1:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for target, data in runs.items():
        results = data["summary"]["restart_results"]
        axes[0].plot(
            [item["restart_index"] for item in results],
            [item["best_objective"] for item in results],
            marker="o",
            label=target.upper(),
        )
    axes[0].set(
        xlabel="Restart",
        ylabel="Best objective",
        title="Best objective by restart",
        yscale="log",
    )
    axes[0].legend()
    axes[0].grid(alpha=0.25)

    labels = [target.upper() for target in runs]
    totals = [data["timing"]["total_seconds"] for data in runs.values()]
    axes[1].bar(labels, totals, color=["tab:orange", "tab:blue"][:len(labels)])
    axes[1].set(ylabel="Wall time (s)", title="Total recorded run time")
    axes[1].grid(axis="y", alpha=0.25)
    display(figure)
    plt.close(figure)
else:
    print("Select RUN_TARGETS = ('cpu', 'gpu') to show the comparison.")